## WaveNet-style CNN

Below is an implementation of a model following the "wave" architecture as seen in Google DeepMind's [WaveNet](https://arxiv.org/pdf/1609.03499) paper. We start with a sequence of 8 tokens, and slowly flatten it to 

In [1]:
from datasets import load_dataset

ds = load_dataset("parquet", 
                    data_files={'train': 'data/train.parquet', 
                                "validation" : "data/validation.parquet", 
                                'test': 'data/test.parquet'}
                    )

c:\Users\n_mac\Desktop\Coding Portfolio\sentencenet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
text_split = []

for row in ds["train"]["sentence"]:
    for w in row.split():
        text_split.append(w)
        
text_split = set(text_split)

In [3]:
s_i = {s:i+1 for i, s in enumerate(text_split)}
s_i["<n>"] = 0

i_s = {i:s for s, i in s_i.items()}

In [4]:
import torch

# building dataset

block_size = 8

def build_dataset(sents):
    
    X, Y = [], []
    for s in sents:
        #print(s)
        context = [0] * block_size
        s[-1] = "<n>"
        
        for w in s:
            
            ix = s_i[w]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [31]:
sents_split = [s.split() for s in ds["train"]["sentence"]]

Xtr, ytr = build_dataset(sents_split)

In [32]:
dstr = torch.utils.data.TensorDataset(Xtr, ytr)

trloader = torch.utils.data.DataLoader(
    dataset=dstr,
    batch_size=500,
    shuffle=True,
    
)

In [39]:
""" class Reshape(torch.nn.Module):    
    def __call__(self, x):
        # print(x.shape[0] / 2)
        if x.dim() == 2:
            return x.view((int(x.shape[0] / 2), -1))
        else:
            return torch.reshape(x, (int(x.shape[0]), int(x.shape[1] / 8), -1))
        
class Squeeze(torch.nn.Module):
    def __call__(self, x):
        return  """

class NN(torch.nn.Module):
    def __init__(self, vocab_size, emb_dim, n_hidden):
        super().__init__()
        
        self.vocab_size = vocab_size
        """ 
        self.layers = [
            torch.nn.Embedding(vocab_size, emb_dim),
            torch.nn.Linear(emb_dim, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Squeeze(), torch.nn.Linear(n_hidden, vocab_size)
        ] """
        
        self.embedding = torch.nn.Embedding(vocab_size, emb_dim)
        self.rnn = torch.nn.RNN(input_size=emb_dim, hidden_size=n_hidden, num_layers=4)
        self.reshape = Reshape()
        self.squeeze = Squeeze()
        self.l1 = torch.nn.Linear(n_hidden * 8, vocab_size)
        
        self.out = 0.0
   
        self.parameters_ = [p for p in self.embedding.parameters()] + [p for p in self.rnn.parameters()] + [p for p in self.l1.parameters()]
        
    """ def parameters(self):
        params = []
        
        for layer in self.layers:
            for p in layer.parameters:
                params.append(p)
        
        return params """
    
    def __call__(self, x, hn):
        
        
        x_ = self.embedding(x)
        # print(x_.shape)
        
        x_, hn = self.rnn(x_, hn)
        # print(x_.shape)
        
        single = True if x_.dim() == 2 else False
        
        x_ = x_.view((int(x_.shape[0] / 2), -1)) if single else torch.reshape(x_, (int(x_.shape[0]), int(x_.shape[1] / 8), -1))
        # print(x_.shape)
        
        x_ = torch.squeeze(x_)
        
        x_ = self.l1(x_)     
        # print(x_.shape)  
        
        self.out = x_
        return x_, hn
    
    def fit(self, max_iter, loader, lr):
        g = torch.Generator().manual_seed(2147483647)
        optimizer = torch.optim.AdamW(self.parameters_, lr=lr)
        
        lossi = []
        
        for p in self.parameters_:
            p.retain_grad()
            
        hn = None
        
        for step in range(max_iter):
            Xb, yb = next(iter(loader))
            
            if hn is not None:
                hn = hn.detach()
            
            logits, hn = self.__call__(Xb, hn)
            
            loss = torch.nn.functional.cross_entropy(logits, yb)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            lossi.append(loss.item())
            
            for p in self.parameters_:
                p = p - lr * p.grad
                
            if step % (max_iter / 10) == 0:
                print(f"{step} / {max_iter}: {loss:.6f}")
        
        return lossi
                

In [44]:
embedding_dim = 32

n_hidden = 100
vocab_size = len(s_i)

net = NN(vocab_size=vocab_size, emb_dim=embedding_dim, n_hidden=n_hidden)


In [ ]:
rnn = torch.nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)

AttributeError: 'Tensor' object has no attribute 'adj'

In [41]:
# net(Xtr[1], None)
net(Xtr[10].clone().expand(1, 8), None)

(tensor([-0.0465, -0.1664, -0.0453,  ..., -0.0918,  0.0325,  0.0265],
        grad_fn=<ViewBackward0>),
 tensor([[[-4.9729e-01,  3.0078e-01,  4.4175e-01,  ..., -8.3893e-02,
           -1.2098e-01,  4.4928e-01],
          [ 3.2098e-01, -3.2161e-01,  3.3629e-01,  ..., -6.9487e-01,
            2.4601e-01,  3.7187e-02],
          [ 4.9318e-01,  1.2814e-01, -1.1823e-01,  ..., -7.7355e-01,
           -3.0907e-01,  2.0279e-01],
          ...,
          [ 2.5820e-01, -3.2912e-01,  5.8928e-01,  ...,  4.8654e-01,
           -5.0021e-01, -5.8773e-01],
          [-5.4917e-02, -1.3075e-01,  5.3448e-01,  ..., -5.5657e-01,
           -3.6247e-01,  3.0282e-01],
          [ 7.6088e-02, -3.2815e-01,  2.8878e-01,  ..., -5.7901e-01,
           -1.6922e-01,  6.5616e-01]],
 
         [[ 2.1671e-01, -1.9102e-01,  1.9307e-01,  ..., -3.1732e-02,
            2.9926e-02, -6.4462e-02],
          [-2.8368e-02, -2.9620e-01, -2.1333e-01,  ...,  1.9955e-04,
            1.0364e-02, -4.4255e-03],
          [-4.4676e-01

In [ ]:
net.fit(max_iter=10000,  loader=trloader, lr=1e-3);

0 / 10000: 9.207117
1000 / 10000: 6.004961
2000 / 10000: 5.650013
3000 / 10000: 4.946173


In [ ]:
Xval, yval = build_dataset(s.split() for s in ds["validation"]["sentence"])
dsval = torch.utils.data.TensorDataset(Xval, yval)

valloader = torch.utils.data.DataLoader(dsval, batch_size=100)

In [ ]:
def eval_model(model, loader):
    total_loss = 0.0
    
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            loss = torch.nn.functional.cross_entropy(logits, y)
            
            total_loss += loss.item() * x.size(0)
            
    return total_loss / len(loader.dataset)

In [ ]:
print("Training set loss:", eval_model(net, trloader))
print("Validation set loss:", eval_model(net, valloader))

Training set loss: 4.836368589899691
Validation set loss: 5.253106556217818


In [ ]:
g2 = torch.Generator().manual_seed(2147483647 + 1)
hni = None

for _ in range(5):
    out = []
    context = [0] * block_size
    
    
    while True:
        probs, hni = net(torch.tensor(context).expand(1, 8), hn=hni)
        logits = torch.nn.functional.softmax(probs, dim=0)

        ix = torch.multinomial(logits, num_samples=1, generator=g2).item()

        context = context[1:] + [ix]
        
        if ix == 0:
            break
        
        out.append(ix)
        
        # if len(out) > 5:
        #     break
        # print(i_s[ix])
        

    print("".join(i_s[i] + " " for i in out))
    

trinity e. largest recruiting silicon their 
adrs length comparison boosted supreme expiration 
loral oversight newsprint slower date premiums 
top electronic initial practiced perjury accountants 
biotechnology not laboratory complete generic kirk 
